# BloomBERT — Focal + Mean Pooling + Label Smoothing

Three-way combination using masked mean pooling (custom head) and a linear
warmup+decay schedule. Ingredients:

- Focal loss (γ=2) with inverse-frequency class weights
- Masked mean pooling over token states
- Label smoothing (ε=0.1)

Solo references on the current data: focal 0.7283, focal+mean 0.7294,
focal+smoothing 0.7304. Everything else (top-3 unfrozen, lr=2e-5, LLRD=0.9,
8 epochs, batch 32, max_len 128, head dropout 0.1, fp16, seed 42) is identical to
the solo runs.

## Setup
1. Attach the `baylearn-bloom-data` dataset. 2. GPU T4 x1. 3. Run All (~35 min).
4. Output `/kaggle/working/bloom_distilbert_focal_meanpool_smoothing/` is zipped at the end.

In [ ]:
# ---- Cell 1: Imports + Config -------------------------------------------
import os, json, random, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,                       # bare encoder; custom mean-pool head
    DistilBertForSequenceClassification,   # only for the HF-compatible copy
    get_linear_schedule_with_warmup,       # linear schedule
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Config = {
    'model_name'      : 'distilbert-base-uncased',
    'max_len'         : 128, 'batch_size': 32, 'epochs': 8,
    'lr'              : 2e-5, 'llrd_decay': 0.9, 'weight_decay': 0.01,
    'warmup_frac'     : 0.10, 'unfreeze_top': 3, 'num_classes': 3,
    'use_amp'         : True,
    'focal_gamma'     : 2.0,     # focal
    'label_smoothing' : 0.1,     # smoothing
    'pooling'         : 'mean',  # mean pooling
    'head_dropout'    : 0.1,
}
LABEL_TO_ID = {'easy': 0, 'medium': 1, 'hard': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

DATA_DIR = '/kaggle/input/datasets/manarabdelshafy/baylearn-bloom-data'
OUT_DIR  = '/kaggle/working/bloom_distilbert_focal_meanpool_smoothing'
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Focal + mean pooling + smoothing (linear schedule)')

In [ ]:
# ---- Cell 2: Data -------------------------------------------------------
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.dropna(subset=['question', 'level'], inplace=True)
    df['level'] = df['level'].str.lower().str.strip()
    df.drop(df[~df['level'].isin(LABEL_TO_ID.keys())].index, inplace=True)
    print(f'  {name}: {len(df)}/{before}')
print('\nLabel distribution in train:'); print(train_df['level'].value_counts())

In [ ]:
# ---- Cell 3: Dataset + loaders ------------------------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained(Config['model_name'])
class BloomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['question'].astype(str).tolist()
        self.labels = [LABEL_TO_ID[l] for l in df['level']]
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                             max_length=self.max_len, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}
train_ds = BloomDataset(train_df, tokenizer, Config['max_len'])
val_ds   = BloomDataset(val_df,   tokenizer, Config['max_len'])
test_ds  = BloomDataset(test_df,  tokenizer, Config['max_len'])
train_loader = DataLoader(train_ds, batch_size=Config['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ---- Cell 4: Custom model with MEAN-POOL head --------------------------
class DistilBertBloomClassifier(nn.Module):
    def __init__(self, model_name, num_classes, pooling='mean', head_dropout=0.1):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size; self.pooling = pooling
        self.pre_classifier = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(head_dropout)
        self.classifier = nn.Linear(hidden, num_classes)
        nn.init.xavier_uniform_(self.pre_classifier.weight); nn.init.zeros_(self.pre_classifier.bias)
        nn.init.xavier_uniform_(self.classifier.weight);     nn.init.zeros_(self.classifier.bias)
    def forward(self, input_ids, attention_mask):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == 'cls':
            pooled = hs[:, 0]
        else:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hs * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        x = F.gelu(self.pre_classifier(pooled)); x = self.dropout(x)
        return self.classifier(x)
def build_model(unfreeze_top):
    model = DistilBertBloomClassifier(Config['model_name'], Config['num_classes'],
                                      pooling=Config['pooling'], head_dropout=Config['head_dropout'])
    for p in model.encoder.parameters(): p.requires_grad = False
    layers = model.encoder.transformer.layer
    for i in range(6 - unfreeze_top, 6):
        for p in layers[i].parameters(): p.requires_grad = True
    for p in model.pre_classifier.parameters(): p.requires_grad = True
    for p in model.classifier.parameters():     p.requires_grad = True
    return model.to(device)
_m = build_model(Config['unfreeze_top'])
total = sum(p.numel() for p in _m.parameters())
train_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total: {total:,}   Trainable: {train_p:,}   ({100*train_p/total:.1f}%)')
del _m; torch.cuda.empty_cache()

In [ ]:
# ---- Cell 5: Class weights + Focal loss WITH label smoothing -----------
counts = train_df['level'].value_counts(); tot = counts.sum()
weights = {l: tot / (Config['num_classes'] * counts[l]) for l in counts.index}
class_weights = torch.tensor([weights[ID_TO_LABEL[i]] for i in range(3)],
                             dtype=torch.float, device=device)
print('Class weights:', {ID_TO_LABEL[i]: round(class_weights[i].item(), 3) for i in range(3)})
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, gamma, alpha, label_smoothing, num_classes):
        super().__init__(); self.gamma=gamma; self.alpha=alpha
        self.eps=label_smoothing; self.K=num_classes
    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1); probs = log_probs.exp()
        with torch.no_grad():
            oh = F.one_hot(targets, num_classes=self.K).float()
            soft = oh*(1-self.eps) + (1-oh)*(self.eps/(self.K-1)) if self.eps>0 else oh
        p_t = (probs*oh).sum(-1); focal = (1-p_t).clamp(min=1e-9).pow(self.gamma)
        ce = -(soft*log_probs).sum(-1)
        return (self.alpha[targets]*focal*ce).mean()

In [ ]:
# ---- Cell 6: LLRD parameter groups (encoder.transformer.layer prefix) --
def build_llrd_parameter_groups(model, base_lr, decay, weight_decay):
    no_decay = ('bias', 'LayerNorm.weight', 'LayerNorm.bias'); groups = []
    for layer_idx in range(6):
        lr = base_lr * (decay ** (5 - layer_idx))
        prefix = f'encoder.transformer.layer.{layer_idx}.'
        dec, nodec = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad or not n.startswith(prefix): continue
            (nodec if any(nd in n for nd in no_decay) else dec).append(p)
        if dec:   groups.append({'params': dec,   'lr': lr, 'weight_decay': weight_decay})
        if nodec: groups.append({'params': nodec, 'lr': lr, 'weight_decay': 0.0})
    hdec, hnodec = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'classifier' not in n and 'pre_classifier' not in n: continue
        (hnodec if any(nd in n for nd in no_decay) else hdec).append(p)
    if hdec:   groups.append({'params': hdec,   'lr': base_lr, 'weight_decay': weight_decay})
    if hnodec: groups.append({'params': hnodec, 'lr': base_lr, 'weight_decay': 0.0})
    return groups

In [ ]:
# ---- Cell 7: Train + eval loops (custom model -> logits) ---------------
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train(); total = 0.0
    for batch in loader:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device, non_blocking=True)
        attn= batch['attention_mask'].to(device, non_blocking=True)
        lab = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast(): logits = model(ids, attn); loss = loss_fn(logits, lab)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(ids, attn); loss = loss_fn(logits, lab)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        scheduler.step(); total += loss.item()
    return total / len(loader)
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); total = 0.0; preds, gold = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        attn= batch['attention_mask'].to(device, non_blocking=True)
        lab = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast(): logits = model(ids, attn)
        else: logits = model(ids, attn)
        total += loss_fn(logits.float(), lab).item()
        preds.extend(logits.argmax(-1).cpu().tolist()); gold.extend(lab.cpu().tolist())
    return total/len(loader), f1_score(gold, preds, average='macro'), preds, gold

In [ ]:
# ---- Cell 8: Artifact-saving helper -------------------------------------
def save_test_artifacts(out_dir, preds, gold, test_loss, test_f1, history, config):
    tn = ['easy', 'medium', 'hard']; os.makedirs(out_dir, exist_ok=True)
    with open(f'{out_dir}/training_history.json', 'w') as f:
        json.dump({'history': history, 'test_macro_f1': test_f1, 'test_loss': test_loss, 'config': config}, f, indent=2)
    with open(f'{out_dir}/test_classification_report.txt', 'w') as f:
        f.write(f'TEST macro_f1 = {test_f1:.4f}   loss = {test_loss:.4f}\n\n')
        f.write(classification_report(gold, preds, target_names=tn, digits=4))
    with open(f'{out_dir}/test_classification_report.json', 'w') as f:
        json.dump(classification_report(gold, preds, target_names=tn, digits=4, output_dict=True), f, indent=2)
    cm = confusion_matrix(gold, preds)
    with open(f'{out_dir}/test_confusion_matrix.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['true \\ pred'] + tn)
        for i, row in enumerate(cm): w.writerow([tn[i]] + list(row))
    with open(f'{out_dir}/label_map.json', 'w') as f: json.dump(LABEL_TO_ID, f, indent=2)

In [ ]:
# ---- Cell 9: MAIN run (focal + mean pooling + smoothing) ---------------
model = build_model(Config['unfreeze_top'])
loss_fn = FocalLossWithSmoothing(Config['focal_gamma'], class_weights,
                                 Config['label_smoothing'], Config['num_classes'])
optimizer = torch.optim.AdamW(build_llrd_parameter_groups(model, Config['lr'], Config['llrd_decay'], Config['weight_decay']))
total_steps = len(train_loader) * Config['epochs']
warmup_steps = int(total_steps * Config['warmup_frac'])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = GradScaler(enabled=Config['use_amp'])
history = []; best_f1, best_state = -1.0, None
for epoch in range(1, Config['epochs'] + 1):
    tr = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    vl, vf1, _, _ = evaluate(model, val_loader, loss_fn)
    history.append({'epoch': epoch, 'train_loss': tr, 'val_loss': vl, 'val_macro_f1': vf1})
    print(f'  epoch {epoch}/{Config["epochs"]}   train={tr:.4f}  val={vl:.4f}  val_f1={vf1:.4f}')
    if vf1 > best_f1: best_f1 = vf1; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
test_loss, test_f1, preds, gold = evaluate(model, test_loader, loss_fn)
print(f'\nTEST  macro_f1={test_f1:.4f}   loss={test_loss:.4f}')
print(classification_report(gold, preds, target_names=['easy','medium','hard'], digits=4))
print('Confusion matrix (rows=true, cols=pred):  easy / medium / hard'); print(confusion_matrix(gold, preds))
save_test_artifacts(OUT_DIR, preds, gold, test_loss, test_f1, history, Config)
print(f'\nTest artifacts saved to {OUT_DIR}')
print('\n=== COMPARISON (current data) ===')
print('  Focal                        : 0.7283')
print('  Focal + mean pooling         : 0.7294')
print('  Focal + label smoothing      : 0.7304')
print(f'  Focal + mean + smoothing     : {test_f1:.4f}  ({test_f1 - 0.7304:+.4f} vs best solo)')

In [ ]:
# ---- Cell 10: Save weights (custom .pt + HF copy) ----------------------
torch.save({'state_dict': model.state_dict(), 'config': Config, 'test_macro_f1': test_f1},
           f'{OUT_DIR}/meanpool_smoothing_state.pt')
hf = DistilBertForSequenceClassification.from_pretrained(Config['model_name'], num_labels=Config['num_classes'])
hf.distilbert.load_state_dict(model.encoder.state_dict())
hf.pre_classifier.weight.data.copy_(model.pre_classifier.weight.data)
hf.pre_classifier.bias.data.copy_(model.pre_classifier.bias.data)
hf.classifier.weight.data.copy_(model.classifier.weight.data)
hf.classifier.bias.data.copy_(model.classifier.bias.data)
hf.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
print('Saved meanpool_smoothing_state.pt (mean-pool) and HF copy.')

In [ ]:
# ---- Cell 11: Sanity check ----------------------------------------------
model.eval()
samples = ['Define a semaphore.',
  'Compute the number of page faults for the reference string 1,2,3,4,1,2,5 under LRU with 3 frames.',
  'Design a deadlock-free dining philosophers protocol using only binary semaphores and justify why it avoids circular wait.']
with torch.no_grad():
    enc = tokenizer(samples, padding=True, truncation=True, max_length=Config['max_len'], return_tensors='pt').to(device)
    logits = model(enc['input_ids'], enc['attention_mask'])
    probs = torch.softmax(logits, -1).cpu().numpy(); preds = logits.argmax(-1).cpu().tolist()
for s, p, pr in zip(samples, preds, probs):
    print(f'\n[{ID_TO_LABEL[p]}]  (easy={pr[0]:.2f}  medium={pr[1]:.2f}  hard={pr[2]:.2f})'); print(f'  Q: {s}')

In [ ]:
# ---- Final cell: ZIP ----------------------------------------------------
import shutil
zp = shutil.make_archive('/kaggle/working/bloom_distilbert_focal_meanpool_smoothing', 'zip', OUT_DIR)
print(f'Created: {zp}  ({os.path.getsize(zp)/1e6:.1f} MB)')
print('Download bloom_distilbert_focal_meanpool_smoothing.zip from the Output panel.')